# Project reference solution — compact execution

Trainer copy. Reveal after completing the guided notebook.

In [ ]:
from pathlib import Path
import json, sys
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / 'training_lib.py').is_file(): ROOT = candidate; break
else: raise FileNotFoundError('Workshop root not found')
sys.path.insert(0, str(ROOT))
from training_lib import create_spark
from project.pipeline import run_pipeline, write_combined_report
spark = create_spark('project-reference-solution')

In [ ]:
results = {}
for run_id in ('run-001', 'run-002', 'run-003'):
    metrics, report = run_pipeline(spark, run_id, ROOT / 'project')
    results[run_id] = metrics
    print(run_id, metrics, report)
combined_report = write_combined_report(spark, ROOT / 'project')
combined_report

In [ ]:
expected = json.loads((ROOT / 'solutions' / 'project_expected_metrics.json').read_text())
for run_id, metrics in results.items():
    assert all(metrics[key] == value for key, value in expected[run_id].items())
base = ROOT / 'project' / 'destination' / 'curated_sales'
destination = spark.read.option('basePath', str(base)).parquet(str(base))
assert destination.count() == expected['all-runs']['accepted_sales_rows']
print('PASS — three runs, Parquet partitions, audits, and reports complete')